# <font color='blue'> Chapter 27: Backpropagation Through Time (BPTT) </font>

In previous chapters, we learned how ordinary backpropagation computes gradients in feedforward neural networks by applying the Chain Rule from the output layer back to the input layer.

Recurrent Neural Networks (RNNs) introduce an additional complication: information flows not only through layers but also through **time**.

Since the hidden state at each time step depends on the hidden state from the previous time step, the loss at one point in time can influence parameters used many time steps earlier.

To train an RNN correctly, gradients must therefore be propagated **backwards through time** as well as through the network itself.

This algorithm is known as **Backpropagation Through Time (BPTT)**.

---

# <font color='orange'> 1. Why Ordinary Backpropagation is Not Enough </font>

Recall the hidden state equation

$$
\boxed{
\mathbf{h}_t
=
f
(
W_{xh}\mathbf{x}_t
+
W_{hh}\mathbf{h}_{t-1}
+
\mathbf{b}
).
}
$$

Notice that

$$
\mathbf{h}_t
$$

depends on

$$
\mathbf{h}_{t-1},
$$

which depends on

$$
\mathbf{h}_{t-2},
$$

and so on.

Therefore,

the loss at time

$$
t
$$

depends on **all previous hidden states**.

---

# <font color='orange'> 2. Unrolling the Network </font>

Instead of viewing the RNN as a loop,

we "unroll" it.

```
Time

t=1

x₁

↓

h₁

↓

y₁

↓

t=2

x₂

↓

h₂

↓

y₂

↓

t=3

x₃

↓

h₃

↓

y₃
```

Although it appears to be several networks,

every copy shares exactly the same parameters

$$
W_{xh},
\;
W_{hh},
\;
W_{hy}.
$$

---

# <font color='orange'> 3. Total Loss </font>

Suppose a sequence has

$$
T
$$

time steps.

Each step contributes its own loss,

$$
L_t.
$$

The total loss becomes

$$
\boxed{
L
=
\sum_{t=1}^{T}
L_t.
}
$$

Training therefore minimizes the loss over the entire sequence,

rather than only the final output.

---

# <font color='orange'> 4. Applying the Chain Rule </font>

Suppose we wish to compute

$$
\frac{\partial L}
{\partial W_{hh}}.
$$

The parameter

$$
W_{hh}
$$

appears repeatedly,

once at every time step.

Therefore,

its gradient is

$$
\boxed{
\frac{\partial L}
{\partial W_{hh}}
=
\sum_{t=1}^{T}
\frac{\partial L}
{\partial \mathbf{h}_t}
\,
\frac{\partial \mathbf{h}_t}
{\partial W_{hh}}.
}
$$

Unlike feedforward networks,

the same parameter contributes to many different parts of the computational graph.

---

# <font color='orange'> 5. Gradients Propagate Through Time </font>

Consider

$$
\mathbf{h}_3.
$$

Its gradient depends on

$$
\mathbf{h}_2,
$$

which depends on

$$
\mathbf{h}_1.
$$

Thus,

the Chain Rule becomes

$$
\boxed{
\frac{\partial L}
{\partial \mathbf{h}_1}
=
\frac{\partial L}
{\partial \mathbf{h}_3}
\,
\frac{\partial \mathbf{h}_3}
{\partial \mathbf{h}_2}
\,
\frac{\partial \mathbf{h}_2}
{\partial \mathbf{h}_1}.
}
$$

The gradient must pass through every intermediate hidden state.

---

# <font color='orange'> 6. Why the Vanishing Gradient Problem Occurs </font>

Suppose each derivative has magnitude

$$
0.8.
$$

After ten time steps,

the gradient becomes

$$
0.8^{10}
=
0.107.
$$

After fifty time steps,

$$
0.8^{50}
\approx
1.4\times10^{-5}.
$$

The gradient is now almost zero.

Consequently,

early time steps receive almost no learning signal.

This phenomenon is called the

**Vanishing Gradient Problem**.

---

# <font color='orange'> 7. Why the Exploding Gradient Problem Occurs </font>

Now suppose

each derivative equals

$$
1.2.
$$

After ten steps,

$$
1.2^{10}
=
6.19.
$$

After fifty steps,

$$
1.2^{50}
\approx
9100.
$$

The gradients become extremely large,

leading to unstable parameter updates.

This is called the

**Exploding Gradient Problem**.

---

# <font color='orange'> 8. Visualising Gradient Flow </font>

Vanishing gradients

```
1

↓

0.8

↓

0.64

↓

0.51

↓

0.41

↓

...
```

Eventually,

the gradient disappears.

---

Exploding gradients

```
1

↓

1.5

↓

2.25

↓

3.38

↓

5.06

↓

...
```

Eventually,

the updates become unstable.

---

# <font color='orange'> 9. Gradient Clipping </font>

A common solution for exploding gradients is

**Gradient Clipping**.

Instead of allowing gradients to become arbitrarily large,

their magnitude is limited.

For example,

if

$$
\|\mathbf{g}\|>5,
$$

replace

$$
\boxed{
\mathbf{g}
\leftarrow
5
\,
\frac{\mathbf{g}}
{\|\mathbf{g}\|}.
}
$$

The gradient direction is preserved,

but its magnitude is reduced.

---

# <font color='orange'> 10. Truncated Backpropagation Through Time </font>

Very long sequences may contain

thousands

of time steps.

Backpropagating through the entire sequence is computationally expensive.

Instead,

training often uses

**Truncated BPTT**.

```
Sequence

□□□□□□□□□□□□□□□□□□

↓

Backpropagate

Only Last

20 Steps
```

The hidden state is carried forward,

but gradients are propagated through only a limited window.

This significantly reduces memory usage and computation.

---

# <font color='orange'> 11. Why LSTMs Were Developed </font>

The vanishing gradient problem makes it difficult for standard RNNs to learn long-range dependencies.

For example,

```
The book that I bought yesterday was extremely interesting because ...

...

...

...

...

it ______
```

To choose the correct verb,

the model may need information from many words earlier.

Standard RNNs often lose this information.

This limitation motivated the development of

- Long Short-Term Memory (LSTM) networks,
- Gated Recurrent Units (GRUs).

---

# <font color='orange'> 12. Practical Implications </font>

Basic RNNs work well for

- short sequences,
- simple temporal patterns.

For longer sequences,

practitioners usually employ

- LSTMs,
- GRUs,
- Transformers.

These architectures address the limitations of basic RNNs.

---

# <font color='red'> 13. Mathematical Foundations </font>

For an unrolled RNN,

the gradient with respect to a recurrent parameter is

$$
\boxed{
\frac{\partial L}
{\partial W_{hh}}
=
\sum_{t=1}^{T}
\left(
\frac{\partial L}
{\partial \mathbf{h}_t}
\prod_{k=1}^{t}
\frac{\partial \mathbf{h}_k}
{\partial \mathbf{h}_{k-1}}
\right)
\frac{\partial \mathbf{h}_t}
{\partial W_{hh}}.
}
$$

Notice the product

$$
\prod
\frac{\partial \mathbf{h}_k}
{\partial \mathbf{h}_{k-1}}.
$$

If these terms are

- less than one,

the product shrinks exponentially.

If they are

- greater than one,

the product grows exponentially.

This repeated multiplication is the fundamental mathematical reason why vanishing and exploding gradients occur in recurrent neural networks.

---

# <font color='orange'> 14. Common Misconceptions </font>

### Misconception 1

> BPTT is a completely different algorithm from Backpropagation.

**False.**

Backpropagation Through Time is an extension of ordinary backpropagation. It still relies on the Chain Rule but applies it across both network layers and time steps.

---

### Misconception 2

> Vanishing gradients occur only in RNNs.

**False.**

Very deep feedforward networks can also suffer from vanishing gradients. However, the problem is often more severe in RNNs because gradients are repeatedly multiplied across many time steps.

---

### Misconception 3

> Gradient clipping fixes the vanishing gradient problem.

**False.**

Gradient clipping limits excessively large gradients, helping with exploding gradients. It does not prevent gradients from becoming too small.

---

# <font color='purple'> 15. Conceptual Summary </font>

| Concept | Description |
|:---|:---|
| Backpropagation Through Time (BPTT) | Extends backpropagation to sequential data |
| Unrolled RNN | Representation of an RNN across multiple time steps |
| Total Sequence Loss | Sum of losses across all time steps |
| Vanishing Gradient | Gradients decrease exponentially over time |
| Exploding Gradient | Gradients increase exponentially over time |
| Gradient Clipping | Limits gradient magnitude to stabilize training |
| Truncated BPTT | Backpropagates through a limited number of time steps |

> **Key Insight:** Backpropagation Through Time trains recurrent neural networks by applying the Chain Rule across both layers and time. Because the same recurrent weights are reused at every time step, gradients accumulate over the entire sequence. The repeated multiplication of derivatives can cause gradients to vanish or explode, making it difficult for standard RNNs to learn long-range dependencies. These challenges motivated the development of gated architectures such as LSTMs and GRUs.